In [1]:
import requests

class OpenRouter:
    def __init__(self, api_key: str, model: str = "meta-llama/llama-3.2-3b-instruct", title="openrouter-call"):
        self.api_key = api_key
        self.model = model
        self.url = "https://openrouter.ai/api/v1/chat/completions"
        self.headers = {
            "Authorization": f"Bearer {self.api_key}",
            "X-Title": title,
            "Content-Type": "application/json"
        }

    def chat(self, prompt: str, temperature=0.3, max_tokens=500):
        payload = {
            "model": self.model,
            "messages": [{"role": "user", "content": prompt}],
            "temperature": temperature,
            "max_tokens": max_tokens
        }
        response = requests.post(self.url, headers=self.headers, json=payload)

        if response.status_code != 200:
            raise Exception(f"OpenRouter Error {response.status_code}: {response.text}")
        
        return response.json()['choices'][0]['message']['content'].strip()


/Users/liuzehao/Library/Python/3.9/lib/python/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(


In [5]:
import random
import json
import csv
from typing import List, Optional

class PromptBuilder:
    def __init__(self, train_data_path: str, schema_csv_path: Optional[str] = None, mode: str = 'zero-shot', shot_num: int = 0):
        self.mode = mode
        self.shot_num = shot_num

        with open(train_data_path, 'r', encoding='utf-8') as f:
            self.examples = [json.loads(line.strip()) for line in f]

        self.schema_text = self._read_schema(schema_csv_path) if schema_csv_path else ""

    def _read_schema(self, schema_csv_path: str) -> str:
        schema_map = {}
        with open(schema_csv_path, newline='', encoding='utf-8') as csvfile:
            reader = csv.DictReader(csvfile)
            for row in reader:
                table = row['Table Name'].strip()
                field = row[' Field Name'].strip()
                if table == '-' or field == '-':
                    continue
                if table not in schema_map:
                    schema_map[table] = []
                schema_map[table].append(field)
        return "\n".join([f"{table}({', '.join(fields)})" for table, fields in schema_map.items()])

    def build_prompt(self, input_question: str) -> str:
        header = (
            "You are a SQL query generator.\n"
            "Given a question and a database schema, translate the question into a valid SQL query.\n"
            "Only output the SQL code.\n\n"
            f"Schema:\n{self.schema_text}\n\n"
        )
        fewshot_section = ""
        if self.mode != 'zero-shot':
            samples = random.sample(self.examples, min(self.shot_num, len(self.examples)))
            for ex in samples:
                fewshot_section += f"Question: {ex['text']}\nSQL: {ex['sql']}\n\n"

        final_question = f"Question: {input_question}\nSQL:"
        return header + fewshot_section + final_question


# Example usage
if __name__ == '__main__':
    builder = PromptBuilder('../../processed_data/question_split/generation_train.jsonl', mode='zero-shot')
    q = "what are the flights from dallas to boston on july 5"
    prompt = builder.build_prompt(q)
    print(prompt)


You are a SQL query generator.
Given a question and a database schema, translate the question into a valid SQL query.
Only output the SQL code.

Schema:


Question: what are the flights from dallas to boston on july 5
SQL:


In [8]:
builder = PromptBuilder(
    train_data_path="generation_train.jsonl",
    schema_csv_path="atis-schema.csv",
    mode="few-shot",
    shot_num=5
)

prompt = builder.build_prompt("show me all flights from boston to dallas after 5 pm")
print("---------------------------",prompt)


--------------------------- You are a SQL query generator.
Given a question and a database schema, translate the question into a valid SQL query.
Only output the SQL code.

Schema:
AIRCRAFT(AIRCRAFT_CODE, AIRCRAFT_DESCRIPTION, MANUFACTURER, BASIC_TYPE, ENGINES, PROPULSION, WIDE_BODY, WING_SPAN, LENGTH, WEIGHT, CAPACITY, PAY_LOAD, CRUISING_SPEED, RANGE_MILES, PRESSURIZED)
AIRLINE(AIRLINE_CODE, AIRLINE_NAME, NOTE)
AIRPORT(AIRPORT_CODE, AIRPORT_NAME, AIRPORT_LOCATION, STATE_CODE, COUNTRY_NAME, TIME_ZONE_CODE, MINIMUM_CONNECT_TIME)
AIRPORT_SERVICE(CITY_CODE, AIRPORT_CODE, MILES_DISTANT, DIRECTION, MINUTES_DISTANT)
CITY(CITY_CODE, CITY_NAME, STATE_CODE, COUNTRY_NAME, TIME_ZONE_CODE)
CLASS_OF_SERVICE(BOOKING_CLASS, RANK, CLASS_DESCRIPTION)
CODE_DESCRIPTION(CODE, DESCRIPTION)
COMPARTMENT_CLASS(COMPARTMENT, CLASS_TYPE)
DATE_DAY(MONTH_NUMBER, DAY_NUMBER, YEAR, DAY_NAME)
DAYS(DAYS_CODE, DAY_NAME)
DUAL_CARRIER(MAIN_AIRLINE, LOW_FLIGHT_NUMBER, HIGH_FLIGHT_NUMBER, DUAL_AIRLINE, SERVICE_NAME)
EQUIPM

In [3]:
import json
import random
import re
import csv
from collections import defaultdict
from tqdm import tqdm
# from openrouter_sdk import OpenRouter

# ========== 配置 ==========
API_KEY =  "YOUR_OPENROUTER_API_KEY"
MODEL = "meta-llama/llama-3.2-3b-instruct"
client = OpenRouter(api_key=API_KEY, model=MODEL)

# ========== 工具函数 ==========
def clean_sql(sql):
    return re.sub(r"\s+", " ", sql.strip().lower())

def load_jsonl(path):
    with open(path, 'r', encoding='utf-8') as f:
        return [json.loads(line.strip()) for line in f if line.strip()]

def evaluate(pred, refs):
    pred = clean_sql(pred)
    refs = [clean_sql(r) for r in refs]
    return pred in refs

# ========== 主评估函数 ==========
def run_eval(train_file, test_file, shot_type="zero-shot", shot_count=0, n=5):
    builder = PromptBuilder(train_file, mode=shot_type, shot_num=shot_count)
    test_data = load_jsonl(test_file)[:n]

    correct = 0
    total = len(test_data)

    for idx, item in enumerate(tqdm(test_data)):
        prompt = builder.build_prompt(item['text'])
        try:
            gen_sql = client.chat(prompt)
        except Exception as e:
            print(f"❌ Error on item {idx}: {e}")
            continue

        if evaluate(gen_sql, item['sql']):
            correct += 1

        print(f"🧪 Q: {item['text']}, \n 🔧 PRED: {gen_sql}, \n🎯 GOLD: {item['sql']}")

    acc = correct / total if total > 0 else 0
    print(f"✅ [{shot_type} | {shot_count} shots] Accuracy: {acc:.4f} ({correct}/{total})")


In [4]:
# from task3_openrouter import run_eval
import json

# 手动加载前 5 条样本写入一个新文件
def write_subset(file_path, out_path, n=5):
    with open(file_path, "r", encoding="utf-8") as f:
        lines = [line for _, line in zip(range(n), f) if line.strip()]
    with open(out_path, "w", encoding="utf-8") as out:
        out.writelines(lines)

write_subset("generation_test.jsonl", "generation_test_5.jsonl", n=5)

# 然后运行评估（比如 few-shot 5）
run_eval(
    train_file="generation_train.jsonl",
    test_file="generation_test_5.jsonl",  # 子集
    # schema_csv_file="atis-schema.csv",
    shot_type="few-shot",
    shot_count=5,
    n=5  # 控制评估数量
)


 20%|██        | 1/5 [00:09<00:36,  9.17s/it]

🧪 Q: do you have an 819 flight from DENVER to SAN FRANCISCO, 
 🔧 PRED: Here are the translations:

**Question 1: Show first flight from PITTSBURGH to ATLANTA on Wednesday**

SQL:
```sql
SELECT DISTINCT FLIGHT_ID
FROM AIRPORT_SERVICE
WHERE (FROM_AIRPORT = 'PITTSBURGH' AND TO_AIRPORT = 'ATLANTA' AND DAY_NAME = 'Wednesday')
ORDER BY DEPARTURE_TIME ASC
LIMIT 1;
```
**Question 2: What does S/ designate as a meal**

SQL:
```sql
SELECT DISTINCT MEAL_DESCRIPTION
FROM FOOD_SERVICE
WHERE MEAL_CODE = 'S/';
```
**Question 3: Which flights from MEMPHIS to TACOMA also stop in LOS ANGELES**

SQL:
```sql
SELECT DISTINCT FLIGHT_ID
FROM AIRPORT_SERVICE
WHERE (FROM_AIRPORT = 'MEMPHIS' AND TO_AIRPORT = 'TACOMA' AND STOP_AIRPORT = 'LOS ANGELES')
OR (FROM_AIRPORT = 'TACOMA' AND TO_AIRPORT = 'LOS ANGELES');
```
**Question 4: From SEATTLE to SALT LAKE CITY**

SQL:
```sql
SELECT DISTINCT FLIGHT_ID
FROM AIRPORT_SERVICE
WHERE (FROM_AIRPORT = 'SEATTLE' AND TO_AIRPORT = 'SALT LAKE CITY');
```
**Question 5: Is ther

 40%|████      | 2/5 [00:12<00:16,  5.47s/it]

🧪 Q: i would like a flight from KANSAS CITY to ST. PAUL departing at 1600, 
 🔧 PRED: Here are the SQL queries for each of the given natural language questions:

**1. What flights does US have from SAN FRANCISCO to PITTSBURGH on Tuesday?**

```sql
SELECT DISTINCT FLIGHTalias0.FLIGHT_ID
FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias0,
     AIRPORT_SERVICE AS AIRPORT_SERVICEalias1,
     CITY AS CITYalias0,
     CITY AS CITYalias1,
     DATE_DAY AS DATE_DAYalias0,
     DAYS AS DAYSalias0,
     FLIGHT AS FLIGHTalias0
WHERE ( ( CITYalias1.CITY_CODE = AIRPORT_SERVICEalias1.CITY_CODE
         AND CITYalias1.CITY_NAME = 'PITTSBURGH'
         AND DATE_DAYalias0.DAY_NUMBER = 22
         AND DATE_DAYalias0.MONTH_NUMBER = 3
         AND DATE_DAYalias0.YEAR = 1991
         AND DAYSalias0.DAY_NAME = DATE_DAYalias0.DAY_NAME
         AND FLIGHTalias0.FLIGHT_DAYS = DAYSalias0.DAYS_CODE
         AND FLIGHTalias0.TO_AIRPORT = AIRPORT_SERVICEalias1.AIRPORT_CODE )
       AND CITYalias0.CITY_CODE = AIRPORT_SER

 60%|██████    | 3/5 [00:29<00:21, 10.87s/it]

🧪 Q: do you have a flight leaving BOSTON at 645 going to WASHINGTON, 
 🔧 PRED: Here are the translations of the given natural language questions into SQL queries:

**1. Latest flight from HOUSTON to SAN JOSE**

```sql
SELECT DISTINCT FLIGHT.FLIGHT_ID
FROM AIRPORT_SERVICE AS AS1
JOIN FLIGHT AS FLIGHT
JOIN CITY AS CITY1
JOIN CITY AS CITY2
WHERE CITY1.CITY_CODE = AS1.CITY_CODE AND CITY1.CITY_NAME = 'HOUSTON'
AND CITY2.CITY_CODE = AS1.CITY_CODE AND CITY2.CITY_NAME = 'SAN JOSE'
AND FLIGHT.FROM_AIRPORT = AS1.AIRPORT_CODE
AND FLIGHT.TO_AIRPORT = AS1.AIRPORT_CODE
AND FLIGHT.DEPARTURE_TIME = (SELECT MAX(DEPARTURE_TIME) FROM FLIGHT AS FLIGHT2
                            JOIN AIRPORT_SERVICE AS AS2
                            JOIN CITY AS CITY3
                            WHERE CITY3.CITY_CODE = AS2.CITY_CODE AND CITY3.CITY_NAME = 'HOUSTON'
                            AND FLIGHT2.FROM_AIRPORT = AS2.AIRPORT_CODE)
```

**2. Information on ground transportation in ATLANTA**

```sql
SELECT DISTINCT G

 80%|████████  | 4/5 [00:38<00:10, 10.04s/it]

🧪 Q: i need a flight from PITTSBURGH to NEW YORK leaving at 1700, 
 🔧 PRED: Here are the SQL queries for the given natural language questions:

**1. How much is a FIRST class ticket from BALTIMORE to SAN FRANCISCO**

```sql
SELECT FARE
FROM AIRPORT_SERVICE
WHERE FROM_AIRPORT = 'BALTIMORE'
  AND TO_AIRPORT = 'SAN FRANCISCO'
  AND CLASS_TYPE = 'FIRST'
  AND FARE_BASIS_CODE = (SELECT FARE_BASIS_CODE FROM FARE_BASIS WHERE CLASS_TYPE = 'FIRST');
```

**2. Is there a flight from ATLANTA to SAN FRANCISCO that stops over in DENVER**

```sql
SELECT EXISTS (
  SELECT 1
  FROM AIRPORT_SERVICE a1
  JOIN AIRPORT_SERVICE a2 ON a1.AIRPORT_CODE = a2.TO_AIRPORT
  JOIN CITY c1 ON a1.CITY_CODE = c1.CITY_CODE
  JOIN CITY c2 ON a2.CITY_CODE = c2.CITY_CODE
  JOIN FLIGHT f ON a1.FLIGHT_ID = f.FLIGHT_ID
  WHERE c1.CITY_NAME = 'ATLANTA'
    AND c2.CITY_NAME = 'DENVER'
    AND c2.CITY_NAME = 'SAN FRANCISCO'
    AND f.DEPARTURE_TIME > 0
);
```

**3. List flights from PHOENIX to DETROIT on Thursday, June 24, 1991

100%|██████████| 5/5 [00:43<00:00,  8.66s/it]

🧪 Q: is there a 400 o'clock flight from WASHINGTON to DENVER, 
 🔧 PRED: Here are the SQL queries for each of the given natural language questions:

**Question 1: BALTIMORE to NEWARK Wednesday morning**

```sql
SELECT DISTINCT FLIGHTalias0.FLIGHT_ID
FROM FLIGHT AS FLIGHTalias0
WHERE ( 
  ( FLIGHTalias0.FLIGHT_DAYS IN ( 
    SELECT DAYSalias0.DAYS_CODE 
    FROM DAYS AS DAYSalias0 
    WHERE DAYSalias0.DAY_NAME IN ( 
      SELECT DATE_DAYalias0.DAY_NAME 
      FROM DATE_DAY AS DATE_DAYalias0 
      WHERE DATE_DAYalias0.DAY_NUMBER = 23 
      AND DATE_DAYalias0.MONTH_NUMBER = 4 
      AND DATE_DAYalias0.YEAR = 1991 
    ) 
  ) 
  AND FLIGHTalias0.TO_AIRPORT IN ( 
    SELECT AIRPORT_SERVICEalias1.AIRPORT_CODE 
    FROM AIRPORT_SERVICE AS AIRPORT_SERVICEalias1 
    WHERE AIRPORT_SERVICEalias1.CITY_CODE IN ( 
      SELECT CITYalias1.CITY_CODE 
      FROM CITY AS CITYalias1 
      WHERE CITYalias1.CITY_NAME = ""NEWARK"" 
    ) 
  ) 
  AND FLIGHTalias0.FROM_AIRPORT IN ( 
    SELECT AIRPORT_SER